# Dining Philosophers: Full-Scale Reproduction Notebook

**Paper:** *Rotating Independent-Set Scheduling for the Dining Philosophers Problem: Replication, Bounded-Wait Analysis, and a Guarded-Fill Adaptive Extension*

This notebook accompanies the paper and is intentionally explicit about prior art. The fixed rotating group round-robin (GRR) schedule is **not claimed as novel**; Lee (2023, DOI `10.7236/JIIBC.2023.23.2.73`) substantially anticipates it. The experiments here reproduce/formalize GRR and evaluate the **Guarded-Fill GRR (GF-GRR)** extension.

The archived CSV/JSON outputs were produced by the scripts included with this artifact. By default this notebook loads those completed outputs so that inspection is fast. Set `RUN_FULL = True` below to regenerate the full core benchmark (720 algorithm-runs; runtime depends strongly on hardware and JIT state).

In [1]:
from pathlib import Path
import json, pandas as pd, numpy as np
ROOT = Path.cwd()
if not (ROOT / 'experiment.py').exists():
    candidate = Path('/mnt/data/dpp_research')
    if candidate.exists(): ROOT = candidate
print('Artifact root:', ROOT)
RUN_FULL = False

Artifact root: /mnt/data/dpp_research


## 1. Formal model and rotating schedule

Philosophers are vertices of the cycle graph $C_N$. A feasible simultaneous set is an independent set. GRR uses

$$S_t=\{(t+2k)\bmod N : k=0,\ldots,\lfloor N/2\rfloor-1\}.$$

The structural checker verifies independence, maximum cardinality, and opportunity-gap behavior for $N=3,\ldots,500$.

In [2]:
from experiment import structural_check
failures, max_gap = structural_check(500)
print({'N_checked': '3..500', 'failures': int(failures), 'maximum_observed_interservice_gap': int(max_gap)})

{'N_checked': '3..500', 'failures': 0, 'maximum_observed_interservice_gap': 3}


## 2. Completed core benchmark

Core design: $N\in\{101,1001\}$, $p\in\{0.05,0.20,0.40\}$, uniform/hotspot demand, 5 schedulers, 12 paired replications, 300 warm-up + 3000 measured rounds. This yields **720 algorithm-runs**.

In [3]:
agg = pd.read_csv(ROOT / 'benchmark_aggregate.csv')
cols=['p','workload','algorithm','throughput_mean','mean_wait_mean','max_wait_max','jain_mean','utilization']
summary=agg[(agg.N==1001) & agg.algorithm.isin(['GRR','GuardedFill','AgeMIS','RotatingGreedy'])][cols]
summary.sort_values(['workload','p','algorithm']).reset_index(drop=True)

,p,workload,algorithm,throughput_mean,mean_wait_mean,max_wait_max,jain_mean,utilization
0,0.05,hotspot,AgeMIS,49.189972,0.110744,4,0.514957,0.098380
1,0.05,hotspot,GRR,47.585806,0.527615,2,0.533640,0.095172
2,0.05,hotspot,GuardedFill,49.123833,0.117910,2,0.516363,0.098248
3,0.05,hotspot,RotatingGreedy,49.194500,0.110594,7,0.514871,0.098389
4,0.20,hotspot,AgeMIS,165.837056,0.347217,5,0.714002,0.331674
5,0.20,hotspot,GRR,160.118917,0.626599,2,0.692450,0.320238
6,0.20,hotspot,GuardedFill,165.754111,0.353547,2,0.713207,0.331508
7,0.20,hotspot,RotatingGreedy,165.674306,0.348777,25,0.712670,0.331349
8,0.40,hotspot,AgeMIS,318.574917,0.488857,5,0.965193,0.637150
9,0.40,hotspot,GRR,303.330111,0.661524,2,0.955591,0.606660


## 3. Primary paired statistics: GuardedFill vs GRR

The primary analysis uses throughput and mean waiting, paired by random seed. The artifact records paired mean differences, 95% confidence intervals, Wilcoxon signed-rank tests, and Holm correction across the 24 planned comparisons.

In [4]:
primary = pd.read_csv(ROOT / 'primary_statistics_guardedfill_vs_grr.csv')
primary.sort_values(['N','p','workload','metric']).reset_index(drop=True)

,N,p,workload,metric,GuardedFill_mean,GRR_mean,diff_mean,ci95_low,ci95_high,paired_t_p,wilcoxon_p,paired_t_p_holm,wilcoxon_p_holm
0,101,0.05,hotspot,mean_wait,0.110406,0.543362,-0.432956,-0.436057,-0.429855,5.442114e-23,0.000488,5.442114e-22,0.011719
1,101,0.05,hotspot,throughput,4.959750,4.791694,0.168056,0.163153,0.172958,2.757014e-16,0.000488,2.757014e-16,0.011719
2,101,0.05,uniform,mean_wait,0.053836,0.525487,-0.471651,-0.474304,-0.468997,3.823171e-24,0.000488,4.970122e-23,0.011719
3,101,0.05,uniform,throughput,5.030694,4.915778,0.114917,0.111890,0.117943,8.972069e-17,0.000488,1.794414e-16,0.011719
4,101,0.20,hotspot,mean_wait,0.358686,0.639321,-0.280634,-0.281830,-0.279439,1.793549e-25,0.000488,2.690323e-24,0.011719
5,101,0.20,hotspot,throughput,16.683667,16.075472,0.608194,0.597354,0.619035,1.229494e-18,0.000488,3.688481e-18,0.011719
6,101,0.20,uniform,mean_wait,0.247435,0.572483,-0.325048,-0.326714,-0.323382,1.371462e-24,0.000488,1.920047e-23,0.011719
7,101,0.20,uniform,throughput,19.211389,18.089028,1.122361,1.109176,1.135546,1.255719e-20,0.000488,6.278594e-20,0.011719
8,101,0.40,hotspot,mean_wait,0.521963,0.676013,-0.154050,-0.155003,-0.153097,1.083903e-23,0.000488,1.192294e-22,0.011719
9,101,0.40,hotspot,throughput,31.851694,30.493917,1.357778,1.341568,1.373988,1.500055e-20,0.000488,6.278594e-20,0.011719


## 4. Stress test ($N=10{,}001$)

In [5]:
stress = pd.read_csv(ROOT / 'stress_10001_aggregate.csv')
stress[['p','workload','algorithm','throughput_mean','mean_wait_mean','max_wait_max','jain_mean']].sort_values(['workload','p','algorithm']).reset_index(drop=True)

,p,workload,algorithm,throughput_mean,mean_wait_mean,max_wait_max,jain_mean
0,0.2,hotspot,AgeMIS,1656.496500,0.347702,6,0.713706
1,0.2,hotspot,GRR,1600.360333,0.625283,2,0.691348
2,0.2,hotspot,GuardedFill,1656.364833,0.353054,2,0.712129
3,0.2,hotspot,RotatingGreedy,1654.970000,0.349063,22,0.714726
4,0.4,hotspot,AgeMIS,3182.762667,0.488666,6,0.964558
5,0.4,hotspot,GRR,3032.177500,0.659967,2,0.955034
6,0.4,hotspot,GuardedFill,3158.723833,0.515119,2,0.963172
7,0.4,hotspot,RotatingGreedy,3184.030000,0.486540,26,0.964633
8,0.2,uniform,AgeMIS,1911.605500,0.230309,5,0.996841
9,0.2,uniform,GRR,1799.743833,0.555633,2,0.997048


## 5. Canonical five-philosopher experiments

Thirty replications and 10,000 measured rounds were used for the original $N=5$ problem size.

In [6]:
n5 = pd.read_csv(ROOT / 'canonical_n5_aggregate.csv')
n5[['p','workload','algorithm','throughput_mean','mean_wait_mean','max_wait_max','jain_mean']].sort_values(['workload','p','algorithm']).reset_index(drop=True)

,p,workload,algorithm,throughput_mean,mean_wait_mean,max_wait_max,jain_mean
0,0.05,hotspot,AgeMIS,0.249567,0.022465,2,0.308260
1,0.05,hotspot,GRR,0.219547,0.870934,2,0.326630
2,0.05,hotspot,GuardedFill,0.249850,0.023904,2,0.308021
3,0.05,hotspot,RotatingGreedy,0.249983,0.024383,3,0.307899
4,0.20,hotspot,AgeMIS,0.938607,0.098292,3,0.317303
5,0.20,hotspot,GRR,0.583123,1.149784,2,0.420638
6,0.20,hotspot,GuardedFill,0.942703,0.120125,2,0.315409
7,0.20,hotspot,RotatingGreedy,0.948390,0.143827,4,0.313183
8,0.40,hotspot,AgeMIS,1.567327,0.463703,6,0.814073
9,0.40,hotspot,GRR,1.244760,1.097499,2,0.916191


## 6. Exact cycle selector verification

The AgeMIS / residual-fill dynamic program optimizes the lexicographic objective `(served_count, summed_waiting_age)` on a cycle. The archived brute-force verification compares it against exhaustive enumeration on 1,500 random cycles of size 3--12.

In [7]:
with open(ROOT / 'age_mis_bruteforce_verification.json') as f:
    print(json.load(f))

{'cases': 1500, 'failures': 0, 'n_range': '3..12', 'objective': 'lexicographic (max cardinality, then total waiting age)'}


## 7. Selector scaling

These are implementation-specific Numba timings for **materialized selectors**, not universal complexity constants. Fixed GRR can be represented implicitly in a production system.

In [8]:
pd.read_csv(ROOT / 'selector_scaling.csv')

,N,algorithm,repetitions,median_ms,mean_ms,p95_ms
0,1000,GRR,30,0.001077,0.001152,0.001529
1,1000,RotatingGreedy,30,0.002294,0.002854,0.004956
2,1000,AgeMIS,30,0.002163,0.002519,0.004550
3,10000,GRR,30,0.009093,0.009114,0.009175
4,10000,RotatingGreedy,30,0.026084,0.031611,0.047726
5,10000,AgeMIS,30,0.035539,0.122063,0.125475
6,100000,GRR,10,0.089539,0.095383,0.109773
7,100000,RotatingGreedy,10,0.478417,0.972618,3.105689
8,100000,AgeMIS,10,0.581282,0.792044,1.720650
9,1000000,GRR,3,1.051015,1.379956,2.041548


## 8. Optional full regeneration

Running the next cell with `RUN_FULL=True` overwrites the core benchmark outputs in `ROOT`. Keep it `False` if you only want to inspect the archived experiment.

In [9]:
if RUN_FULL:
    from experiment import run_bench
    run_bench(str(ROOT))
else:
    print('RUN_FULL=False: using archived completed results.')

RUN_FULL=False: using archived completed results.


## 9. Key interpretation

- GRR gives a deterministic conflict-free service schedule and maximum saturated concurrency $\lfloor N/2\rfloor$.
- In the synchronous model, even $N$ has a 2-round service-opportunity gap; odd $N$ has 2- or 3-round gaps.
- Fixed GRR can waste capacity under dynamic demand.
- GF-GRR keeps every hungry philosopher selected by the GRR guard, then fills residual nonconflicting capacity adaptively.
- In the tested workloads, GF-GRR preserved the GRR tail bound while recovering most of the throughput achieved by unconstrained adaptive schedulers.

These conclusions are **model-specific**. They do not establish superiority over asynchronous/distributed protocols such as Chandy--Misra, nor historical priority for the general idea of age-aware or independent-set scheduling.